# 1.install and import packges 

In [2]:
!pip install -qU \
    langchain \
    langchain-community \
    langchain-groq \
    pypdf \
    fastembed \
    chromadb \
    -qU langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 10.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 59.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━

In [3]:
import langchain
import langchain_groq
import pypdf
import fastembed
import chromadb
from langchain_community.document_loaders import PyPDFLoader
import os

print("All packages installed successfully!")

All packages installed successfully!


/tmp/ipykernel_48/481779447.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


=======================================================\
pypdf : to load a pdf \
fastembde : to use the embedding \
chromdb : to store the embedding 


# 2.import the llm and the embedding model 

In [4]:
from kaggle_secrets import UserSecretsClient
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_groq import ChatGroq
import os
#llm
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY


class Models:
    def __init__(
        self,
        Chat_model="openai/gpt-oss-20b",
        embedding_model="thenlper/gte-large"
    ):
        self.Chat_model = Chat_model
        self.embedding_model_name = embedding_model

        self.chat_llm = ChatGroq(
            model=self.Chat_model,
            temperature=0.2,
            max_tokens=4000,
        )

        self.embedding_model = FastEmbedEmbeddings(
            model_name=self.embedding_model_name
        )

# load the pdf 

In [5]:
class File_Loader:
    ALLOWED_FILES = ["pdf"]

    def __init__(self, File_links):
        self.File_links = File_links

    def get_file_type(self, file_link):
        extension = os.path.splitext(file_link)[1].lower().replace(".", "")

        if extension in self.ALLOWED_FILES:
            return extension

        return "none"

    def load_files(self):
        files = []

        for file_link in self.File_links:
            file_type = self.get_file_type(file_link)

            files.append({
                "path": file_link,
                "type": file_type
            })

        return files

    def load_allowed_files(self):
        documents = []

        files = self.load_files()

        for file in files:
            if file["type"] == "pdf":
                loader = PyPDFLoader(file["path"])
                documents.extend(loader.load())

        return documents

file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

loader = File_Loader(file_links)

documents = loader.load_allowed_files()

print(f"Loaded documents: {len(documents)}")

Loaded documents: 11


# splitting the file 

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class Chunker:
    def __init__(self, File_links, chunk_size=1000, chunk_overlap=150):

        # Chunker internally uses File_Loader
        self.file_loader = File_Loader(File_links)

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def split(self):
        # Load allowed files
        documents = self.file_loader.load_allowed_files()

        # Split documents into chunks
        return self.splitter.split_documents(documents)

file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx"
]

chunker = Chunker(file_links)

chunks = chunker.split()

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 27


# embedding 

In [7]:
from langchain_chroma import Chroma

class Embedding:
    def __init__(
        self,
        File_links,
        embedding_model="thenlper/gte-large",
        chat_model="openai/gpt-oss-20b",
        persist_directory="./chroma_db",
        chunk_size=800,
        chunk_overlap=150
    ):
        self.File_links = File_links
        self.persist_directory = persist_directory

        # Use LLM class
        self.Models = Models(
            Chat_model=chat_model,
            embedding_model=embedding_model
        )

        # Get embedding model from LLM
        self.embedding_model = self.Models.embedding_model

        # Get chat model from LLM
        self.chat_llm = self.Models.chat_llm

        # Chunker internally uses File_Loader
        self.chunker = Chunker(
            File_links,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        # Create chunks
        self.chunks = self.chunker.split()

        # Create ChromaDB
        self.db = Chroma.from_documents(
            documents=self.chunks,
            embedding=self.embedding_model,
            persist_directory=self.persist_directory
        )

    def get_db(self):
        return self.db

    def get_chunks(self):
        return self.chunks

    def get_Models(self):
        return self.chat_Models


In [8]:
query = "What is the importance of the title 'I Sell My Dreams ?"

# retriever part [we will use the mmr]

In [9]:

from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder


class Retriever:

    def __init__(
        self,
        embedding,
        documents,
        search_type="mmr",
        k=5,                  # Final number of chunks
        candidate_k=20,       # Candidates from vector search
        bm25_k=20,            # Candidates from BM25
        reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        rrf_k=60
    ):
        self.db = embedding.get_db()

        self.k = k
        self.rrf_k = rrf_k

        # 1. Semantic / vector retriever
        self.retriever = self.db.as_retriever(
            search_type=search_type,
            search_kwargs={"k": candidate_k}
        )

        # 2. BM25 retriever
        self.bm25_retriever = BM25Retriever.from_documents(
            documents
        )
        self.bm25_retriever.k = bm25_k

        # 3. Cross-encoder reranker
        self.reranker = CrossEncoder(reranker_model)

    def invoke(self, query):

        # Vector search
        semantic_docs = self.retriever.invoke(query)

        # Keyword search
        bm25_docs = self.bm25_retriever.invoke(query)

        # RRF fusion: keep all unique candidates
        candidates = self._rrf(
            semantic_docs,
            bm25_docs
        )

        if not candidates:
            return []

        # Cross-encoder scores query-document pairs
        pairs = [
            (query, doc.page_content)
            for doc in candidates
        ]

        scores = self.reranker.predict(pairs)

        # Sort by reranker score, highest first
        ranked_docs = sorted(
            zip(candidates, scores),
            key=lambda item: item[1],
            reverse=True
        )

        # Return only the final K documents
        return [
            doc for doc, score in ranked_docs[:self.k]
        ]

    def _rrf(
        self,
        semantic_docs,
        bm25_docs
    ):
        scores = {}
        documents = {}

        # Semantic results
        for rank, doc in enumerate(semantic_docs, start=1):
            key = self._document_key(doc)

            scores[key] = scores.get(key, 0) + (
                1 / (self.rrf_k + rank)
            )
            documents[key] = doc

        # BM25 results
        for rank, doc in enumerate(bm25_docs, start=1):
            key = self._document_key(doc)

            scores[key] = scores.get(key, 0) + (
                1 / (self.rrf_k + rank)
            )
            documents[key] = doc

        # Return every unique candidate in RRF order
        ranked_keys = sorted(
            scores,
            key=scores.get,
            reverse=True
        )

        return [
            documents[key]
            for key in ranked_keys
        ]

    def _document_key(self, doc):
        return (
            doc.page_content,
            str(sorted(doc.metadata.items()))
        )


# RAG 

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


class RAG_Retrieval:

    def __init__(self, file_links):
        # VectorDB is created ONCE
        self.embedding = Embedding(
            File_links=file_links
        )

    def get_retriever(self, k=5):
        retriever = self.embedding.get_db().as_retriever(
            search_type="mmr",
            search_kwargs={"k": k}
        )

        return retriever




class RAG_Generation:

    def __init__(
        self,
        retrieval,
        k=8,
        template="""
You are an AI assistant that answers questions using the provided context.

Rules:

- If the context does not contain enough information to answer the question,
  say exactly: "I don't know."
- if you have any information that can help say it 
- try to use all context to answer 

Context:
{context}

Question:
{query}

Answer:
"""
    ):

        self.retrieval = retrieval
        self.k = k

        self.models = Models()

        self.prompt = ChatPromptTemplate.from_template(template)

        self.retriever = self.retrieval.get_retriever(
            k=self.k
        )

        self.chain = (
            {
                "context": self.retriever,
                "query": RunnablePassthrough()
            }
            | self.prompt
            | self.models.chat_llm
            | StrOutputParser()
        )

    def response(self, query):
        return self.chain.invoke(query)


In [11]:
file_links = [
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.pdf",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/lekl101.txt",
    "/kaggle/input/datasets/mohamedjahha/pdf-test/test.docx",
    "/kaggle/input/datasets/mohamedjahha/test22/IBMSIQDeploymentConfigGuide76021.pdf"
]

In [12]:
rag_retrieval = RAG_Retrieval(file_links)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [13]:
rag = RAG_Generation(
    rag_retrieval,
)

print(rag.response(
    "What is the significance of the title 'I Sell My Dreams'?"
))

The title “I Sell My Dreams” is a literal and figurative statement that frames the whole story.  
* **Literal sense** – The narrator (and the woman, Frau Frieda, who is central to the plot) actually trades her dreams for money. The title is a direct declaration of that trade.  
* **Metaphorical sense** – It signals the theme of the narrative: the commodification of something as intangible as a dream, and the idea that one can “sell” hope, imagination, or future possibilities in order to survive.  
* **Narrative hook** – By putting the first‑person claim in the title, the author invites the reader to follow the narrator’s journey of turning dreams into a livelihood, and to question what is lost or gained in that exchange.  

Thus the title is significant because it encapsulates the story’s central act—selling dreams—and the broader commentary on how dreams can be turned into a commodity for survival.


In [15]:
import re, time, unicodedata

def norm(s):
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[\u2010-\u2015\-]", " ", s)   # dashes/hyphens -> space
    s = re.sub(r"[’‘`'*_]", "", s)             # apostrophes, markdown
    s = s.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", s).strip()

IDK = ["i dont know", "not in", "no information", "not mentioned", "cannot", "not provided"]

tests = [
    ("At what time and where was the narrator when the wave hit?", ["nine", "9", "havana riviera"]),
    ("How many floors did the Havana Riviera Hotel have?", ["twenty", "20"]),
    ("Who was the woman found in the car in the wall?", ["housekeeper"]),
    ("What jewelry did she wear?", ["serpent", "snake", "emerald"]),
    ("How did Frau Frieda get her name?", ["students"]),
    ("What did she answer when asked how she came to be in Vienna?", ["sell my dreams"]),
    ("How did the first dream she interpreted end?", ["caramel", "choked"]),
    ("What did Frau Frieda tell the narrator the night she told him to leave Vienna?", ["five years"]),
   
]

passed = 0
for q, kw in tests:
    time.sleep(15)
    a = norm(str(rag.response(q)))
    idk = any(p in a for p in IDK)
    if not a:
        status = "EMPTY"
    elif kw:
        status = "PASS" if any(k in a for k in kw) else ("MISS (idk)" if idk else "WRONG")
    else:
        status = "PASS" if idk else "HALLUCINATED"
    passed += status == "PASS"
    print(f"{status:12} | {q}\n             -> {a[:120]}\n")

print(f"Score: {passed}/{len(tests)}")

PASS         | At what time and where was the narrator when the wave hit?
             -> at nine oclock in the morning, on the terrace of the havana riviera hotel.

PASS         | How many floors did the Havana Riviera Hotel have?
             -> twenty floors.

PASS         | Who was the woman found in the car in the wall?
             -> the woman was the housekeeper of the new portuguese ambassador and his wife.

PASS         | What jewelry did she wear?
             -> she wore a snake shaped ring an egyptian serpent ring studded with emerald eyes.

PASS         | How did Frau Frieda get her name?
             -> she never revealed her real name. the latin american students in vienna, who called her a germanic tongue twister, inven

MISS (idk)   | What did she answer when asked how she came to be in Vienna?
             -> i dont know.

PASS         | How did the first dream she interpreted end?
             -> the first dream she interpreted ended tragically the boy choked on a p

In [16]:
import re, time, unicodedata

def norm(s):
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[\u2010-\u2015\-]", " ", s)   # dashes/hyphens -> space
    s = re.sub(r"[’‘`'*_]", "", s)             # apostrophes, markdown
    s = s.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", s).strip()

IDK = ["i dont know", "not in", "no information", "not mentioned", "cannot", "not provided"]

tests = [

    ("What was the name of Neruda's wife?", ["matilde"]),
    ("What did Neruda say about prophetic dreams?", ["poetry"]),
    ("On which finger was the snake ring in Barcelona?", ["index"]),
    ("Which writer did the narrator say Neruda's dream was like?", ["borges"]),
    ("What color ink did Neruda use?", ["green"]),
    ("What did the ambassador say Frau Frieda did?", ["dreamed", "nothing"]),
    ("What is the significance of the title 'I Sell My Dreams'?", ["sell my dreams", "dream"]),
    # Unanswerable
    ("What is Neruda's favorite color?", None),
    ("What was Frau Frieda's real name?", None),
    ("What year does the Havana disaster take place?", None),
    ("What was the title of the book Neruda bought at Porter?", None),
]

passed = 0
for q, kw in tests:
    time.sleep(15)
    a = norm(str(rag.response(q)))
    idk = any(p in a for p in IDK)
    if not a:
        status = "EMPTY"
    elif kw:
        status = "PASS" if any(k in a for k in kw) else ("MISS (idk)" if idk else "WRONG")
    else:
        status = "PASS" if idk else "HALLUCINATED"
    passed += status == "PASS"
    print(f"{status:12} | {q}\n             -> {a[:120]}\n")

print(f"Score: {passed}/{len(tests)}")

PASS         | What was the name of Neruda's wife?
             -> matilde.

PASS         | What did Neruda say about prophetic dreams?
             -> he said he did not believe in prophetic dreams, adding that “only poetry is clairvoyant.”

MISS (idk)   | On which finger was the snake ring in Barcelona?
             -> i dont know.

PASS         | Which writer did the narrator say Neruda's dream was like?
             -> borges.

PASS         | What color ink did Neruda use?
             -> green ink.

PASS         | What did the ambassador say Frau Frieda did?
             -> he said that frau frieda did nothing at all she simply dreamed.

PASS         | What is the significance of the title 'I Sell My Dreams'?
             -> the title “i sell my dreams” is a literal and symbolic hinge of the story. literal meaning the central character, frau f

HALLUCINATED | What is Neruda's favorite color?
             -> green.

HALLUCINATED | What was Frau Frieda's real name?
             -> h

In [17]:
import re, time, unicodedata

def norm(s):
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[\u2010-\u2015\-]", " ", s)   # dashes/hyphens -> space
    s = re.sub(r"[’‘`'*_]", "", s)             # apostrophes, markdown
    s = s.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", s).strip()

IDK = ["i dont know", "not in", "no information", "not mentioned", "cannot", "not provided"]
tests = [
    ("What are the components of the IBM StoredIQ solution?", ["gateway", "data server"]),
    ("What does the gateway do?", ["between the data servers", "application stack"]),
    ("What database does a DataServer - Classic use to store its index?", ["postgres"]),
    ("Where does a DataServer - Distributed store its index?", ["elasticsearch"]),
    ("How many vCPUs does the application stack need?", ["4"]),
    ("What is the default data disk size on a DataServer - Classic?", ["2 tb"]),
    ("What are the general size limits for a classic data server?", ["150 million", "500"]),
    ("How much index storage does metadata need per TB of source data?", ["30 gb"]),
    ("What minimum IOPS gives acceptable data server performance?", ["650"]),
    ("How much memory does each Elasticsearch node need?", ["32"]),
    ("How many gateways are recommended per data servers?", ["50"]),
    ("Which port must be open from the data server to the gateway?", ["11103"]),
    ("Which ports must be open between the gateway and the application stack?", ["8765", "5432"]),
    ("What port does Elasticsearch use for REST requests?", ["9200"]),
    ("How is IBM StoredIQ licensed?", ["rvu", "resource value unit"]),
    ("Which FIPS level does IBM StoredIQ support?", ["level 1"]),
    ("Which disk encryption does Elasticsearch use?", ["luks"]),
    ("What are the default credentials for the superadmin account?", ["admin"]),
    ("In what order should components be configured?", ["elasticsearch", "gateway"]),
    ("Is Microsoft Hyper-V supported for the Elasticsearch appliance?", ["not supported", "no"]),
    ("How many nodes does the default Elasticsearch cluster have?", ["three", "3"]),
    ("Which property is mandatory in cluster-setup.properties?", ["es_hosts"]),
    ("Which option is added to the setup command for a single node?", ["single-node", "single node"]),
    ("What is the minimum cluster size from which you can remove a node?", ["four", "4"]),
    ("Which command restarts services on data servers after changing the Elasticsearch node list?", ["deepfiler"]),
    # Unanswerable
    ("What is the price of an IBM StoredIQ license per RVU?", None),
    ("What is the release date of version 7.6.0.22?", None),
    ("How many concurrent users can the application stack support?", None),
]
passed = 0
for q, kw in tests:
    time.sleep(15)
    a = norm(str(rag.response(q)))
    idk = any(p in a for p in IDK)
    if not a:
        status = "EMPTY"
    elif kw:
        status = "PASS" if any(k in a for k in kw) else ("MISS (idk)" if idk else "WRONG")
    else:
        status = "PASS" if idk else "HALLUCINATED"
    passed += status == "PASS"
    print(f"{status:12} | {q}\n             -> {a[:120]}\n")

print(f"Score: {passed}/{len(tests)}")

PASS         | What are the components of the IBM StoredIQ solution?
             -> the ibm storediq solution is made up of the following components: 1. application stack the front end service that users 

PASS         | What does the gateway do?
             -> the gateway is the component that connects the data server tier to the application stack tier. it handles all inter comp

PASS         | What database does a DataServer - Classic use to store its index?
             -> it uses an embedded postgresql database to store its index.

PASS         | Where does a DataServer - Distributed store its index?
             -> a dataserver distributed stores its index in an elasticsearch cluster.

PASS         | How many vCPUs does the application stack need?
             -> the application stack requires 4 vcpus.

WRONG        | What is the default data disk size on a DataServer - Classic?
             -> the default data disk size on a dataserver classic is 75 gb.

PASS         | What are